# Diabetic Retinopathy — Training Control Panel

- Dataset: `data/btgraham-300/`, preserving the existing `train`, `validation`, and `test` splits.
- Common model input: `300 × 300` RGB; the shared loader performs the resize.
- Current experiment block: EfficientNetB0 baseline (frozen ImageNet backbone + five-class head).
- Recovery checkpoints and best-validation model are stored separately for each run.

## 1. Environment and device check

In [1]:
from configs import tensorflow_setup as _tensorflow_setup

import sys
import numpy as np
import tensorflow as tf

_tensorflow_setup.configure_gpu_memory_growth(tf)

import tensorflow_datasets as tfds

print("Python:", sys.version.split()[0])
print("CUDA DLL directory:", _tensorflow_setup.CUDA_DLL_DIRECTORY)
print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "not exposed")
print("NumPy:", np.__version__)
print("TFDS:", tfds.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("EfficientNetB0 API:", hasattr(tf.keras.applications, "EfficientNetB0"))
print("Seed utility API:", hasattr(tf.keras.utils, "set_random_seed"))

Python: 3.10.20
CUDA DLL directory: C:\Users\kawal\miniconda3\envs\dr_tf\Library\bin
TensorFlow: 2.10.1
Keras: 2.10.0
NumPy: 1.23.5
TFDS: 4.8.3
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
EfficientNetB0 API: True
Seed utility API: True


## 2. Data-loader smoke test

In [2]:
from scripts.data_loader import inspect_dataset, make_datasets

inspect_dataset()
train_ds, validation_ds, test_ds = make_datasets(batch_size=4)
images, labels = next(iter(train_ds))

print("Image batch shape:", images.shape)
print("Image dtype:", images.dtype)
print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))
print("Labels:", labels.numpy())

Dataset directory: D:\M L\Diabetic_Retinopathy_Detection_mini_project\data\btgraham-300
Dataset name:      diabetic_retinopathy_detection
Dataset version:   3.0.0
Features:          ['label', 'image', 'name']
Available splits:
  sample: 10 examples
  train: 35,126 examples
  validation: 10,906 examples
  test: 42,670 examples
Image feature:     Image(shape=(None, None, 3), dtype=uint8)
Label feature:     ClassLabel(shape=(), dtype=int64, num_classes=5)
Train     : train (35,126 examples)
Validation: validation (10,906 examples)
Test      : test (42,670 examples)
Image batch shape: (4, 300, 300, 3)
Image dtype: <dtype: 'float32'>
Pixel range: 0.0 to 255.00003051757812
Labels: [0 0 0 0]


## 3. EfficientNet — model initialization preview

In [3]:
from scripts.train_efficientnet import build_efficientnet_model

efficientnet_preview = build_efficientnet_model(weights=None)
efficientnet_preview.summary()
print("Input:", efficientnet_preview.input_shape)
print("Output:", efficientnet_preview.output_shape)
del efficientnet_preview
tf.keras.backend.clear_session()

Model: "efficientnet_b0_dr"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 retinal_image (InputLayer)  [(None, 300, 300, 3)]     0         
                                                                 
 efficientnetb0 (Functional)  (None, 10, 10, 1280)     4049571   
                                                                 
 global_average_pooling (Glo  (None, 1280)             0         
 balAveragePooling2D)                                            
                                                                 
 head_dropout (Dropout)      (None, 1280)              0         
                                                                 
 dr_severity (Dense)         (None, 5)                 6405      
                                                                 
Total params: 4,055,976
Trainable params: 6,405
Non-trainable params: 4,049,571
__________________________________

## 4. EfficientNet — manually controlled training
Choose `scratch` for a new isolated run or `resume` to continue the newest valid checkpoint. The epoch input is the **total target epoch**, not the number of additional epochs.

In [5]:
from scripts.train_efficientnet import train_efficientnet

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    cpu_choice = input("To deliberately allow potentially slow CPU training, type RUN-ON-CPU: ").strip()
    allow_cpu = cpu_choice == "RUN-ON-CPU"

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, or rerun and explicitly opt into CPU training.")
else:
    run_mode = input("Run mode [scratch/resume]: ").strip().lower()
    batch_text = input("Batch size [4]: ").strip()
    epochs_text = input("Total target epochs [10]: ").strip()

    batch_size = int(batch_text or 4)
    target_epochs = int(epochs_text or 10)

    learning_rate = float(input("Enter value for learning rate: (Default : 0.001) "))

    efficientnet_result = train_efficientnet(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate
    )

Training device(s): /physical_device:GPU:0
Starting a new, separate run: run_001
Train     : train (35,126 examples)
Validation: validation (10,906 examples)
Test      : test (42,670 examples)
Input size: 300x300; classes: 5
Starting at epoch 0; target total epochs: 35
Epoch 1/35
2196/2196 [==============================] - 234s 100ms/step - loss: 0.7742 - accuracy: 0.7456 - val_loss: 0.7123 - val_accuracy: 0.7613
Epoch 2/35
2196/2196 [==============================] - 219s 100ms/step - loss: 0.7384 - accuracy: 0.7530 - val_loss: 0.7020 - val_accuracy: 0.7635
Epoch 3/35
2196/2196 [==============================] - 215s 98ms/step - loss: 0.7311 - accuracy: 0.7534 - val_loss: 0.7005 - val_accuracy: 0.7658
Epoch 4/35
2196/2196 [==============================] - 216s 98ms/step - loss: 0.7278 - accuracy: 0.7549 - val_loss: 0.6918 - val_accuracy: 0.7651
Epoch 5/35
2196/2196 [==============================] - 216s 98ms/step - loss: 0.7237 - accuracy: 0.7551 - val_loss: 0.6864 - val_accuracy: 

In [6]:
from scripts.utils import print_run_status

print_run_status("efficientnet")

Model: efficientnet
run_001: status=completed, epoch=35/35, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet\runs\run_001


[{'run_id': 'run_001',
  'status': 'completed',
  'last_completed_epoch': 35,
  'target_epochs': 35,
  'checkpoint': 'yes',
  'best_model': 'yes',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\efficientnet\\runs\\run_001'}]


## EfficientNetB0 — Baseline

| Parameter | Current Setting |
|---|---|
| Model Architecture | EfficientNetB0 |
| Pretrained Weights | ImageNet |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbone Trainable | False (Frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy |
| Training Metric | Sparse Categorical Accuracy |
| Dropout Rate | 0.30 (30%) |
| Classification Head | Global Average Pooling → Dropout → Dense (5, Softmax) |
| Class Weighting | None (Unweighted Baseline) |
| Batch Size | 16 |
| Target Epochs | 0 |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest Validation Loss |


## 6. Class-weighted EfficientNet — manually controlled training

**Run only after the unweighted EfficientNet baseline experiment you intend to compare against.** This is a separate experiment with its own run folders and checkpoints; it does not initialize from the trained unweighted model. It uses the same EfficientNetB0/ImageNet architecture and the same official `btgraham-300` splits.

Balanced class weights are calculated from the official **training split only** using `w_c = N / (K × n_c)`. They are applied to training examples only. Validation loss remains unweighted for model selection, and the test split is not evaluated here. Choose `scratch` for an independent new run or `resume` to continue the latest weighted-run checkpoint. The Epoch value is the **total target epoch**, not the number of additional epochs.


In [8]:
from scripts.train_efficientnet_weighted import train_efficientnet_weighted

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    cpu_choice = input("To deliberately allow potentially slow CPU training, type RUN-ON-CPU: ").strip()
    allow_cpu = cpu_choice == "RUN-ON-CPU"

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, or rerun and explicitly opt into CPU training.")
else:
    run_mode = input("Run mode [scratch/resume]: ").strip().lower()
    batch_text = input("Batch size [4]: ").strip()
    epoch_text = input("Total target Epoch [10]: ").strip()

    batch_size = int(batch_text or 4)
    target_epochs = int(epoch_text or 10)

    learning_rate = float(input("Enter value for learning rate: (Default : 0.001) "))

    weighted_efficientnet_result = train_efficientnet_weighted(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate
    )

Training device(s): /physical_device:GPU:0
Resuming weighted run run_001 from: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet_weighted\checkpoints\run_001\ckpt-47
Train     : train (35,126 examples)
Validation: validation (10,906 examples)
Test      : test (42,670 examples)
Input size: 300x300; classes: 5
Training-only class counts: {0: 25810, 1: 2443, 2: 5292, 3: 873, 4: 708}
Training-only class weights: {0: 0.27218907400232467, 1: 2.875644699140401, 2: 1.3275132275132275, 3: 8.047193585337915, 4: 9.922598870056497}
Validation weighting: none; best model selected by lowest validation loss
Starting at epoch 47; target total epochs: 50
Epoch 48/50
2196/2196 [==============================] - 219s 97ms/step - loss: 1.1580 - accuracy: 0.4328 - val_loss: 1.2337 - val_accuracy: 0.4258
Epoch 49/50
2196/2196 [==============================] - 206s 93ms/step - loss: 1.1699 - accuracy: 0.4309 - val_loss: 1.2062 - val_accuracy: 0.4487
Epoch 50/50
2196/2196 [==============

In [10]:
from scripts.utils import print_run_status

print_run_status("efficientnet_weighted")

Model: efficientnet_weighted
run_001: status=completed, epoch=50/50, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet_weighted\runs\run_001


[{'run_id': 'run_001',
  'status': 'completed',
  'last_completed_epoch': 50,
  'target_epochs': 50,
  'checkpoint': 'yes',
  'best_model': 'yes',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\efficientnet_weighted\\runs\\run_001'}]

### Class-weighted EfficientNet — current hyperparameters

| Parameter | Current Setting |
|---|---|
| Model Architecture | EfficientNetB0 |
| Pretrained Weights | ImageNet |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbone Trainable | False (Frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy with per-example sample weights |
| Training Metric | Sparse Categorical Accuracy (unweighted reporting metric) |
| Dropout Rate | 0.30 (30%) |
| Classification Head | Global Average Pooling → Dropout → Dense (5, Softmax) |
| Class Weighting | Balanced inverse-frequency: \(w_c = N/(K n_c)\), calculated from the official training split only |
| Batch Size | 4 (default; configurable per run) |
| Epoch | User-defined total target (prompt default: 10) |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest unweighted validation loss |


## 7. Weighted DenseNet121 — preview and training

This experiment uses a frozen ImageNet-pretrained DenseNet121 backbone, global average pooling, dropout, and a five-class softmax head. Balanced inverse-frequency class weights are computed from the official **training split only** and applied to training loss only. Validation remains unweighted. It has independent checkpoints and run folders under `models/densenet/`; it does not load EfficientNet weights. The total target Epoch defaults to 10 and batch size defaults to 4.

In [4]:
from scripts.train_densenet import build_densenet_model

densenet_preview = build_densenet_model(weights=None)
densenet_preview.summary()
print("Input:", densenet_preview.input_shape)
print("Output:", densenet_preview.output_shape)
del densenet_preview
tf.keras.backend.clear_session()

Model: "densenet121_dr_weighted"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 retinal_image (InputLayer)  [(None, 300, 300, 3)]     0         
                                                                 
 densenet_preprocessing (Lam  (None, 300, 300, 3)      0         
 bda)                                                            
                                                                 
 densenet121 (Functional)    (None, 9, 9, 1024)        7037504   
                                                                 
 global_average_pooling (Glo  (None, 1024)             0         
 balAveragePooling2D)                                            
                                                                 
 head_dropout (Dropout)      (None, 1024)              0         
                                                                 
 dr_severity (Dense)         (None, 5)     

### Weighted DenseNet — manually controlled training

Only run this cell when ready to train. If the model runs out of GPU memory, reduce batch size before changing the image size.

In [5]:
from scripts.train_densenet import train_densenet

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False
if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    cpu_choice = input("To deliberately allow potentially slow CPU training, type RUN-ON-CPU: ").strip()
    allow_cpu = cpu_choice == "RUN-ON-CPU"

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, or explicitly opt into CPU training.")
else:
    run_mode = 'scratch'# input("Run mode [scratch/resume]: ").strip().lower()
    batch_text = 4 #input("Batch size [4]: ").strip()
    epoch_text = 50 #input("Total target Epoch [10]: ").strip()
    batch_size = int(batch_text or 4)
    target_epochs = int(epoch_text or 10)
    learning_rate = 0.001 # float(input("Enter value for learning rate: (Default : 0.001) "))
    densenet_result = train_densenet(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate
    )

Training device(s): /physical_device:GPU:0
Starting a new, separate run: densenet/run_001
29084464/29084464 [==============================] - 4s 0us/step
Train     : train (35,126 examples)
Validation: validation (10,906 examples)
Test      : test (42,670 examples)
Input size: 300x300; classes: 5
Training-only class counts: {0: 25810, 1: 2443, 2: 5292, 3: 873, 4: 708}
Training-only class weights: {0: 0.27218907400232467, 1: 2.875644699140401, 2: 1.3275132275132275, 3: 8.047193585337915, 4: 9.922598870056497}
Validation remains unweighted; best model selected by lowest validation loss.
Starting at epoch 0; total target epochs: 50
Epoch 1/50
8782/8782 [==============================] - 496s 55ms/step - loss: 1.5731 - accuracy: 0.3088 - val_loss: 1.2613 - val_accuracy: 0.4615
Epoch 2/50
8782/8782 [==============================] - 478s 54ms/step - loss: 1.5082 - accuracy: 0.3428 - val_loss: 1.0595 - val_accuracy: 0.5604
Epoch 3/50
8782/8782 [==============================] - 478s 54ms/st

### Weighted DenseNet — inspect run status

Lists only this model's runs; it does not load a model or evaluate it.

In [6]:
from scripts.utils import print_run_status

print_run_status("densenet")

Model: densenet
run_001: status=completed, epoch=50/50, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\densenet\runs\run_001


[{'run_id': 'run_001',
  'status': 'completed',
  'last_completed_epoch': 50,
  'target_epochs': 50,
  'checkpoint': 'yes',
  'best_model': 'yes',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\densenet\\runs\\run_001'}]

### Weighted DenseNet121 — hyperparameters

| Parameter | Current Setting |
|---|---|
| Model Architecture | DenseNet121 |
| Pretrained Weights | ImageNet |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbone Trainable | False (Frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy with training sample weights |
| Training Metric | Sparse Categorical Accuracy (unweighted reporting metric) |
| Dropout Rate | 0.30 (30%) |
| Classification Head | DenseNet preprocessing → Global Average Pooling → Dropout → Dense (5, Softmax) |
| Class Weighting | Balanced inverse-frequency: `N / (K × n_c)`, training split only |
| Batch Size | 4 (default; configurable per run) |
| Epoch | User-defined total target (prompt default: 10) |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest unweighted validation loss |

## 8. EfficientNetB0 + DenseNet121 feature-fusion model

The starter fusion experiment concatenates the two branches' global-average-pooled features, followed by dropout and a five-class softmax head. Both ImageNet feature extractors are frozen for this first resource-conscious version. This is **feature concatenation**, not attention fusion and not averaging final predictions. This initial fusion run is unweighted. Because it contains two backbones, batch size 4 may exceed 4 GB VRAM; reduce the batch size first if needed.

In [7]:
from scripts.train_fusion import build_fusion_model

fusion_preview = build_fusion_model(weights=None)
fusion_preview.summary()
print("Input:", fusion_preview.input_shape)
print("Output:", fusion_preview.output_shape)
del fusion_preview
tf.keras.backend.clear_session()

Model: "efficientnet_densenet_fusion_dr"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 retinal_image (InputLayer)     [(None, 300, 300, 3  0           []                               
                                )]                                                                
                                                                                                  
 densenet_preprocessing (Lambda  (None, 300, 300, 3)  0          ['retinal_image[0][0]']          
 )                                                                                                
                                                                                                  
 efficientnetb0 (Functional)    (None, 10, 10, 1280  4049571     ['retinal_image[0][0]']          
                                )                                   

### Fusion — manually controlled training

Training is separate from the EfficientNet and DenseNet run roots. Use `scratch` for a new independent fusion run or `resume` to restore its own latest valid checkpoint.

In [9]:
from scripts.train_fusion import train_fusion

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False
if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    cpu_choice = input("To deliberately allow potentially slow CPU training, type RUN-ON-CPU: ").strip()
    allow_cpu = cpu_choice == "RUN-ON-CPU"

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, or explicitly opt into CPU training.")
else:
    run_mode = input("Run mode [scratch/resume]: ").strip().lower()
    batch_text = input("Batch size [4]: ").strip()
    epoch_text = input("Total target Epoch [10]: ").strip()
    batch_size = int(batch_text or 4)
    target_epochs = int(epoch_text or 10)
    learning_rate = float(input("Enter value for learning rate: (Default : 0.001) "))
    fusion_result = train_fusion(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate
    )

Training device(s): /physical_device:GPU:0
Starting a new, separate run: fusion/run_001
Train     : train (35,126 examples)
Validation: validation (10,906 examples)
Test      : test (42,670 examples)
Class weighting: none; validation remains unweighted.
Input size: 300x300; classes: 5
Best weights are selected using unweighted validation loss.
Starting at epoch 0; total target epochs: 40
Epoch 1/40
2196/2196 [==============================] - 531s 238ms/step - loss: 0.8012 - accuracy: 0.7388 - val_loss: 0.7088 - val_accuracy: 0.7613
Epoch 2/40
2196/2196 [==============================] - 518s 235ms/step - loss: 0.7582 - accuracy: 0.7493 - val_loss: 0.6961 - val_accuracy: 0.7690
Epoch 3/40
2196/2196 [==============================] - 526s 239ms/step - loss: 0.7499 - accuracy: 0.7507 - val_loss: 0.7500 - val_accuracy: 0.7660
Epoch 4/40
2196/2196 [==============================] - 526s 239ms/step - loss: 0.7480 - accuracy: 0.7522 - val_loss: 0.6985 - val_accuracy: 0.7659
Epoch 5/40
2196/2

### Fusion — inspect run status

In [10]:
 from scripts.utils import print_run_status

print_run_status("fusion")

Model: fusion
run_001: status=completed, epoch=40/40, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\fusion\runs\run_001


[{'run_id': 'run_001',
  'status': 'completed',
  'last_completed_epoch': 40,
  'target_epochs': 40,
  'checkpoint': 'yes',
  'best_model': 'yes',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\fusion\\runs\\run_001'}]

### Fusion — current hyperparameters

| Parameter | Current Setting |
|---|---|
| Model Architecture | EfficientNetB0 + DenseNet121 feature concatenation |
| Pretrained Weights | ImageNet for both backbones |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbones Trainable | False (both frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy |
| Training Metric | Sparse Categorical Accuracy |
| Dropout Rate | 0.30 (30%) |
| Classification Head | Concatenate pooled features → Dropout → Dense (5, Softmax) |
| Class Weighting | None (initial fusion experiment) |
| Batch Size | 4 (default; configurable per run) |
| Epoch | User-defined total target (prompt default: 10) |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest unweighted validation loss |
